In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# NB_Gold_Dim_Customer
# Silver -> Gold
# ============================================================

SOURCE_TABLE = "silver_customers"
TARGET_TABLE = "dim_customer"

# ------------------------------------------------------------
# 1. Read Silver
# ------------------------------------------------------------

df = spark.table(SOURCE_TABLE)

# ------------------------------------------------------------
# 2. Select business attributes
# ------------------------------------------------------------

df = (
    df
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "country",
        "signup_date"
    )
)

# ------------------------------------------------------------
# 3. Generate surrogate key
# ------------------------------------------------------------

window = Window.orderBy("customer_id")

df_gold = (
    df
    .withColumn(
        "customer_key",
        F.row_number().over(window)
    )
    .select(
        "customer_key",
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "country",
        "signup_date"
    )
)

# ------------------------------------------------------------
# 4. Write Gold dimension
# ------------------------------------------------------------

df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(TARGET_TABLE)

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

print(f"Gold table created: {TARGET_TABLE}")
print(f"Records: {df_gold.count()}")

df_gold.show(10, truncate=False)


StatementMeta(, 5355b152-79aa-44ab-bb68-513b4c29b48a, 3, Finished, Available, Finished, False)

Gold table created: dim_customer
Records: 9996
+------------+-----------+----------+---------+----------------------------+--------------+-----------+
|customer_key|customer_id|first_name|last_name|email                       |country       |signup_date|
+------------+-----------+----------+---------+----------------------------+--------------+-----------+
|1           |1          |Danielle  |Johnson  |john21@example.net          |Canada        |2025-12-14 |
|2           |2          |Amy       |Garza    |johnsonjeffery@example.net  |United Kingdom|2025-07-15 |
|3           |3          |Anna      |Baldwin  |lrobinson@example.com       |Italy         |2026-02-28 |
|4           |4          |Adam      |Shaffer  |jpeterson@example.org       |Canada        |2026-08-09 |
|5           |5          |Matthew   |Davis    |howardmaurice@example.com   |Germany       |2026-02-25 |
|6           |6          |Jessica   |Herrera  |smiller@example.net         |United States |2025-08-02 |
|7           |7  